### Data Source and Limitations\n\n**Note:** The paper uses a proprietary dataset from Finaeon, which includes 1, 2, 5, and 10-year government bond rates for all ten currencies. As this dataset is not publicly available, this notebook uses public data sources as a substitute:\n\n- **FX Rates:** Pulled from Yahoo Finance.\n- **Bond Yields:** Pulled from the FRED API.\n\nThis leads to a key limitation: the FRED database provides 1, 2, 5, and 10-year yields only for USD. For all other currencies, only the 10-year yield is consistently available. The code adapts to this by using the 10-year yield as a proxy where shorter-term yields are required for non-USD currencies.

# FX Rate Prediction with Graph Learning\n\nThis notebook implements the data fetching and preprocessing steps described in the paper \"Graph Learning for Foreign Exchange Rate Prediction and Statistical Arbitrage\" (arXiv:2508.14784). It prepares the data for a graph-based machine learning model to predict foreign exchange rates.\n\nThe notebook is designed to be run on Google Colab.

## 0. Setup

In [ ]:
!pip install -q yfinance pandas numpy requests scikit-learn torch torch_geometric

## 1. Data Fetching\n\nIn this section, we fetch the necessary data from our two sources:\n\n- **FRED (Federal Reserve Economic Data):** For government bond yields.\n- **Yahoo Finance:** For foreign exchange (FX) rates.

In [ ]:
import yfinance as yf\nimport pandas as pd\nimport numpy as np\nimport requests\nfrom datetime import datetime

### 1.1. FRED Data for Bond Yields

In [ ]:
FRED_API_KEY = '8f503a1e7348fa967987e5ad187992b9'\n\nBOND_SERIES = {\n    'USD': {'1Y': 'DGS1', '2Y': 'DGS2', '5Y': 'DGS5', '10Y': 'DGS10'},\n    'EUR': {'10Y': 'IRLTLT01EZM156N'},\n    'JPY': {'10Y': 'IRLTLT01JPM156N'},\n    'GBP': {'10Y': 'IRLTLT01GBM156N'},\n    'AUD': {'10Y': 'IRLTLT01AUM156N'},\n    'CAD': {'10Y': 'IRLTLT01CAM156N'},\n    'CHF': {'10Y': 'IRLTLT01CHM156N'},\n    # 'HKD': {'10Y': 'IRLTLT01HKM156N'}, # Removed - Data unavailable from FRED
    # 'SGD': {'10Y': 'IRLTLT01SGM156N'}, # Removed - Data unavailable from FRED
    'SEK': {'10Y': 'IRLTLT01SEM156N'}\n}

In [ ]:
def fetch_fred_data(series_id, api_key, start_date='1995-01-01', end_date='2024-12-31'):\n    url = f'https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={api_key}&file_type=json&observation_start={start_date}&observation_end={end_date}'\n    response = requests.get(url)\n    data = response.json()\n    df = pd.DataFrame(data['observations'])\n    df = df[['date', 'value']]\n    df['date'] = pd.to_datetime(df['date'])\n    df = df.set_index('date')\n    df['value'] = pd.to_numeric(df['value'], errors='coerce')\n    return df

### 1.2. Yahoo Finance Data for FX Rates

In [ ]:
CURRENCIES = ['EUR', 'JPY', 'GBP', 'AUD', 'CAD', 'CHF', 'SEK'] # Removed HKD, SGD\nFX_TICKERS = [f'{currency}USD=X' for currency in CURRENCIES] + ['USDJPY=X']

In [ ]:
def fetch_fx_data(tickers, start_date='1995-01-01', end_date='2024-12-31'):\n    data = yf.download(tickers, start=start_date, end=end_date)['Close']\n    return data

### 1.3. Execute Data Fetching

In [ ]:
bond_data = {}\nfor currency, series in BOND_SERIES.items():\n    bond_data[currency] = {}\n    for term, series_id in series.items():\n        print(f'Fetching {currency} {term} bond yield...')\n        bond_data[currency][term] = fetch_fred_data(series_id, FRED_API_KEY)\n\nprint('\nFetching FX rates...')\nfx_data = fetch_fx_data(FX_TICKERS)\n\nprint('\nBond Data:')\nfor currency, data in bond_data.items():\n    for term, df in data.items():\n        print(f'{currency} {term}: {df.shape}')\n\nprint('\nFX Data:')\nprint(fx_data.shape)\nprint(fx_data.head())

## 2. Data Preprocessing and Feature Engineering\n\nIn this section, we preprocess the raw data to make it suitable for our graph model. The main steps are:\n\n1.  **Resampling:** The bond yield data for non-USD currencies is monthly. We need to upsample it to a daily frequency to match the FX data. We will use forward-filling for this.\n2.  **Merging:** Combine the bond yields and FX rates into a single DataFrame.\n3.  **Handling Missing Values:** Fill any remaining missing values.

In [ ]:
daily_index = pd.date_range(start=fx_data.index.min(), end=fx_data.index.max(), freq='D')\n\nprocessed_bond_data = {}\nfor currency, terms in bond_data.items():\n    for term, df in terms.items():\n        col_name = f'{currency}_{term}_yield'\n        # Reindex and forward-fill\n        processed_bond_data[col_name] = df.reindex(daily_index).ffill()\n\nbond_yields_df = pd.concat(processed_bond_data, axis=1)\n\nprint('Processed Bond Yields Data:')\nprint(bond_yields_df.shape)\nprint(bond_yields_df.head())

In [ ]:
df_full = pd.concat([fx_data, bond_yields_df], axis=1)\ndf_full = df_full.ffill()\n\nprint('Shape before cleaning:', df_full.shape)\n\n# Specific data cleaning as mentioned in the paper\n# Note: The dates and values might not exactly match due to using public data.\ndf_full.loc['2014-12-29':'2015-05-20', 'SGDAUD=X'] = np.nan # Repeated entries\ndf_full.loc[df_full['SEKEUR=X'] < 0.069, 'SEKEUR=X'] = np.nan # Erroneous value\ndf_full.loc[df_full.index.isin(['2023-04-24', '2024-01-12', '2024-01-26']) & (df_full['AUDCHF=X'] > 0.85), 'AUDCHF=X'] = np.nan # Erroneous values\ndf_full.loc['2022-01-01':'2023-12-31', ['HKD_10Y_yield', 'SGD_10Y_yield']] = df_full.loc['2022-01-01':'2023-12-31', ['HKD_10Y_yield', 'SGD_10Y_yield']].apply(lambda x: np.where(x > 90, np.nan, x))\n\ndf_full = df_full.ffill()\ndf_full = df_full.dropna()\n\nprint('Shape after cleaning:', df_full.shape)\nprint(df_full.head())

## 3. Time-Series Graph Preparation\n\nThis section implements the feature engineering process (`h_PI`) described in the paper to create a sequence of spatiotemporal graphs. Each graph in the sequence represents the state of the market at a specific time `t`, with features computed from look-back windows.

In [ ]:
from scipy.linalg import lstsq\nimport torch\nfrom torch_geometric.data import Data\n\ndef calculate_currency_values(data_t, currencies):\n    currency_map = {name: i for i, name in enumerate(currencies)}\n    num_currencies = len(currencies)\n    edges = []\n    for i, c1 in enumerate(currencies):\n        for j, c2 in enumerate(currencies):\n            if i < j:\n                # Need to construct the ticker and check if it exists\n                ticker1 = f'{c1}{c2}=X' if c2 != 'USD' else f'{c1}USD=X'\n                if c1 == 'USD': ticker1 = f'{c2}USD=X'\n                if ticker1 not in data_t.index and f'{c2}{c1}=X' in data_t.index:\n                    ticker1 = f'{c2}{c1}=X'\n                if ticker1 in data_t.index and not pd.isna(data_t[ticker1]):\n                    edges.append((c1, c2))\n\n    num_edges = len(edges)\n    A = np.zeros((num_edges + 1, num_currencies))\n    b = np.zeros(num_edges + 1)\n\n    for i, (c1, c2) in enumerate(edges):\n        A[i, currency_map[c1]] = 1\n        A[i, currency_map[c2]] = -1\n        ticker = f'{c1}{c2}=X' if c2 != 'USD' else f'{c1}USD=X'\n        if c1 == 'USD': ticker = f'{c2}USD=X'\n        if ticker not in data_t.index and f'{c2}{c1}=X' in data_t.index:\n            b[i] = -np.log(data_t[f'{c2}{c1}=X'])\n        else:\n            b[i] = np.log(data_t[ticker])\n\n    A[num_edges, :] = 1\n    b[num_edges] = 0\n\n    log_V, _, _, _ = lstsq(A, b)\n    return pd.Series(np.exp(log_V), index=currencies)\n\ndef get_all_fx_tickers(currencies):\n    tickers = []\n    for i in range(len(currencies)):\n        for j in range(i + 1, len(currencies)):\n            c1, c2 = currencies[i], currencies[j]\n            # yfinance uses specific conventions\n            if f'{c1}{c2}=X' in fx_data.columns:\n                tickers.append(f'{c1}{c2}=X')\n            elif f'{c2}{c1}=X' in fx_data.columns:\n                tickers.append(f'{c2}{c1}=X')\n    return list(set(tickers))\n\ndef generate_graph_sequence(df, currencies, lookback_windows):\n    graph_sequence = []\n    # Ensure we have all the tickers we might need for targets\n    all_currencies_inc_usd = currencies + ['USD']\n    df_v = pd.DataFrame({t: calculate_currency_values(df.loc[t], all_currencies_inc_usd) for t in df.index}).T\n\n    # We need t+1 to calculate the target, so we stop one day earlier\n    for t in range(max(lookback_windows), len(df) - 1):\n        # ... (The existing code for node_features and edge_features remains the same)\n        \n        # --- Start of existing code ---\n        # Node Features\n        node_features = []\n        for currency in all_currencies_inc_usd:\n            y_features = []\n            v_features = []\n            for window in lookback_windows:\n                ir_col = f'{currency}_1Y_yield' if currency == 'USD' else f'{currency}_10Y_yield'\n                if ir_col in df.columns:\n                    ir_log_diff = np.log(1 + df[ir_col].iloc[t-window:t] / 100).diff().mean()\n                    y_features.append(ir_log_diff)\n                else:\n                    y_features.append(0)\n                \n                v_log_diff = np.log(df_v[currency].iloc[t-window:t]).diff().mean()\n                v_features.append(v_log_diff)\n            node_features.append(np.concatenate([np.nan_to_num(y_features), np.nan_to_num(v_features)]).flatten())\n        node_features = torch.tensor(node_features, dtype=torch.float)\n        \n        # Edge Features\n        edge_features = []\n        num_nodes = len(all_currencies_inc_usd)\n        edge_index_list = []\n        \n        # --- NEW: Store tickers in same order as edges for easy target lookup ---\n        edge_tickers_list = []\n\n        for i in range(num_nodes):\n            for j in range(num_nodes):\n                if i == j: continue\n                edge_index_list.append([i, j])\n                \n                c1 = all_currencies_inc_usd[i]\n                c2 = all_currencies_inc_usd[j]\n                \n                ticker = f'{c1}{c2}=X'\n                # Handle yfinance ticker conventions\n                if f'{c2}{c1}=X' in df.columns and ticker not in df.columns:\n                    ticker = f'{c2}{c1}=X'\n\n                edge_tickers_list.append(ticker) # Store the ticker\n\n                x_feature = []\n                for window in lookback_windows:\n                    if ticker in df.columns:\n                        fx_log_diff = np.log(df[ticker].iloc[t-window:t]).diff().mean()\n                        x_feature.append(fx_log_diff)\n                    else:\n                        x_feature.append(0)\n                edge_features.append(np.nan_to_num(x_feature))\n\n        edge_features = torch.tensor(edge_features, dtype=torch.float)\n        edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()\n        # --- End of existing code ---\n\n        # --- MODIFICATION: Calculate the actual target 'y' ---\n        y_target = []\n        current_rates = df.iloc[t]\n        next_day_rates = df.iloc[t+1]\n        \n        for ticker in edge_tickers_list:\n            if ticker in df.columns:\n                 # h_PO^-1 from the paper: log(X_t / X_{t-1})\n                log_return = np.log(next_day_rates[ticker]) - np.log(current_rates[ticker])\n                y_target.append(log_return)\n            else:\n                 # If ticker doesn't exist, target is 0\n                y_target.append(0)\n        \n        y = torch.tensor(y_target, dtype=torch.float).view(-1, 1)\n        # --- End MODIFICATION ---\n\n        graph_sequence.append(Data(x=node_features, edge_index=edge_index, edge_attr=edge_features, y=y, edge_tickers=edge_tickers_list, timestamp=df.index[t]))\n    \n    return graph_sequence\n\nlookback_windows = [1, 3, 5, 10, 15, 20]\ngraph_sequence = generate_graph_sequence(df_full, currencies, lookback_windows)\nprint(f"Generated {len(graph_sequence)} graphs for the time series.")\nif graph_sequence:\n    print("Example graph data object:")\n    print(graph_sequence[0])

## 4. Spatiotemporal GNN Model Implementation\n\nThis section implements the spatiotemporal Graph Neural Network (GNN) for FX rate prediction, as described in Section 4.1 of the paper. This model replaces the simple placeholder GNN.

In [ ]:
import torch.nn as nn\nimport torch.nn.functional as F\nfrom torch_geometric.nn import MessagePassing

In [ ]:
class SLP(nn.Module):\n    def __init__(self, in_features, out_features):\n        super(SLP, self).__init__()\n        self.linear = nn.Linear(in_features, out_features)\n        self.leaky_relu = nn.LeakyReLU()\n\n    def forward(self, x):\n        return self.leaky_relu(self.linear(x))

In [ ]:
class FXRP_GNN_Layer(MessagePassing):\n    def __init__(self, node_in_dim, edge_in_dim, node_out_dim, edge_out_dim):\n        super(FXRP_GNN_Layer, self).__init__(aggr='mean')\n        self.slp_node = SLP(node_in_dim + edge_in_dim + node_in_dim, node_out_dim)\n        self.slp_edge = SLP(node_out_dim + edge_in_dim + node_out_dim, edge_out_dim)\n\n    def forward(self, x, edge_index, edge_attr):\n        # Node update (propagate messages)\n        node_features_new = self.propagate(edge_index, x=x, edge_attr=edge_attr)\n        # Edge update\n        row, col = edge_index\n        edge_features_new = self.slp_edge(torch.cat([node_features_new[row], edge_attr, node_features_new[col]], dim=1))\n        return node_features_new, edge_features_new\n\n    def message(self, x_i, x_j, edge_attr):\n        # x_i: target node features, x_j: source node features\n        tmp = torch.cat([x_i, edge_attr, x_j], dim=1)\n        return self.slp_node(tmp)

In [ ]:
class FXRP_GNN(nn.Module):\n    def __init__(self, num_node_features, num_edge_features, hidden_dim, num_layers):\n        super(FXRP_GNN, self).__init__()\n        self.node_embedding = nn.Linear(num_node_features, hidden_dim)\n        self.edge_embedding = nn.Linear(num_edge_features, hidden_dim)\n\n        self.layers = nn.ModuleList()\n        for _ in range(num_layers):\n            self.layers.append(FXRP_GNN_Layer(hidden_dim, hidden_dim, hidden_dim, hidden_dim))\n\n        self.final_slp = nn.Linear(hidden_dim, 1) # No activation for the final layer\n\n    def forward(self, data):\n        node_features, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr\n\n        # Initial embedding\n        node_emb = self.node_embedding(node_features)\n        edge_emb = self.edge_embedding(edge_attr)\n\n        # GNN layers\n        for layer in self.layers:\n            node_emb, edge_emb = layer(node_emb, edge_index, edge_emb)\n\n        # Final prediction\n        return self.final_slp(edge_emb)\n\n# Instantiate the model (example hyperparameters)\nif graph_sequence:\n    model = FXRP_GNN(num_node_features=graph_sequence[0].num_node_features, \n                     num_edge_features=graph_sequence[0].num_edge_features, \n                     hidden_dim=64, \n                     num_layers=3)\n    print("Spatiotemporal GNN Model Architecture:")\n    print(model)

## 5. FXSA Graph and Feature Preparation\n\nThis section implements the feature engineering (`h_SI`) for the second GNN model (`f_S`), which is used for statistical arbitrage. It creates a new graph where nodes are the currency exchanges themselves.

In [ ]:
from scipy.linalg import null_space\nfrom itertools import permutations\n\ndef generate_fxsa_graph_sequence(predictions_sequence, all_currencies, lookback_windows, o='USD', epsilon_S=1e-4):\n    # ... (The existing code for calculating P_hat_history remains the same)\n    # --- Start of existing code ---\n    fxsa_graph_sequence = []\n    u_nodes = list(permutations(all_currencies, 2))\n    u_map = {name: i for i, name in enumerate(u_nodes)}\n    num_u_nodes = len(u_nodes)\n\n    alpha_hat_history = []\n    P_hat_history = []\n\n    for t in range(len(predictions_sequence)):\n        predictions_t = predictions_sequence.iloc[t]\n        # Make sure all_currencies includes USD for value calculation\n        V_hat_t = calculate_currency_values(predictions_t, all_currencies)\n        alpha_hat_t = {}\n        for i,j in u_nodes:\n            ticker = f'{i}{j}=X' if j != 'USD' else f'{i}USD=X'\n            if i == 'USD': ticker = f'{j}USD=X'\n            if ticker in predictions_t:\n                alpha_hat_t[(i,j)] = np.log(predictions_t[ticker]) - np.log(V_hat_t[i]) + np.log(V_hat_t[j])\n            else: alpha_hat_t[(i,j)] = 0\n        alpha_hat_history.append(alpha_hat_t)\n\n        constraint_matrix = []\n        for i in all_currencies:\n            if i == o: continue\n            row = np.zeros(num_u_nodes)\n            for j in all_currencies:\n                if i != j: row[u_map[(i,j)]] = 1\n            constraint_matrix.append(row)\n        for i_idx, i in enumerate(all_currencies):\n            for j_idx, j in enumerate(all_currencies):\n                if i_idx < j_idx:\n                    row = np.zeros(num_u_nodes)\n                    X_hat_toi = predictions_t.get(f'{o}{i}=X'.replace('USDUSD','USD'), 1.0) if i != o else 1.0\n                    X_hat_toj = predictions_t.get(f'{o}{j}=X'.replace('USDUSD','USD'), 1.0) if j != o else 1.0\n                    \n                    ticker_ji = f'{j}{i}=X'.replace('USDUSD','USD')\n                    ticker_ij = f'{i}{j}=X'.replace('USDUSD','USD')\n                    if ticker_ji in predictions_t:\n                        X_hat_ji = predictions_t[ticker_ji]\n                    elif ticker_ij in predictions_t and predictions_t[ticker_ij] != 0:\n                        X_hat_ji = 1.0 / predictions_t[ticker_ij]\n                    else:\n                        X_hat_ji = 1.0 \n\n                    row[u_map[(i,j)]] = X_hat_toi\n                    row[u_map[(j,i)]] = X_hat_toj * X_hat_ji\n                    constraint_matrix.append(row)\n        \n        B_hat_t = null_space(np.array(constraint_matrix))\n        if B_hat_t.size == 0: # Handle case where null space is empty\n             P_hat_t = torch.zeros((num_u_nodes, num_u_nodes), dtype=torch.float)\n        else:\n             P_hat_t = torch.tensor(B_hat_t @ np.linalg.pinv(B_hat_t.T @ B_hat_t) @ B_hat_t.T, dtype=torch.float)\n        P_hat_history.append(P_hat_t)\n    # --- End of existing code ---\n    \n    for t in range(max(lookback_windows), len(predictions_sequence)):\n        # ... (The existing code for node_features and edge_features remains the same)\n        # --- Start of existing code ---\n        # Node features are temporal averages of alpha_hat\n        node_features = []\n        for i,j in u_nodes:\n            alpha_features = []\n            for window in lookback_windows:\n                avg_alpha = np.mean([alpha_hat_history[t-k][(i,j)] for k in range(window)])\n                alpha_features.append(avg_alpha)\n            node_features.append(alpha_features)\n        node_features = torch.tensor(node_features, dtype=torch.float)\n\n        # Edge features are temporal averages of P_hat entries\n        P_hat_t = P_hat_history[t]\n        edge_index = (P_hat_t.abs() > epsilon_S).nonzero().t()\n        edge_features = []\n        for src, dest in edge_index.t():\n            p_features = []\n            for window in lookback_windows:\n                # Taking the mean of a tensor slice\n                avg_p = torch.mean(torch.stack([P_hat_history[t-k][src, dest] for k in range(window)])).item()\n                p_features.append(avg_p)\n            edge_features.append(p_features)\n        edge_features = torch.tensor(edge_features, dtype=torch.float)\n        # --- End of existing code ---\n\n        # --- MODIFICATION: Attach P_hat_t to the data object ---\n        fxsa_graph_sequence.append(Data(x=node_features, \n                                        edge_index=edge_index, \n                                        edge_attr=edge_features,\n                                        P_hat=P_hat_history[t], # Attach the matrix\n                                        timestamp=predictions_sequence.index[t],\n                                        u_nodes=u_nodes\n                                        ))\n        # --- End MODIFICATION ---\n    \n    return fxsa_graph_sequence\n\ndummy_predictions = df_full[get_all_fx_tickers(currencies + ['USD'])]\nfxsa_graph_sequence = generate_fxsa_graph_sequence(dummy_predictions, currencies + ['USD'], lookback_windows)\nprint(f"Generated {len(fxsa_graph_sequence)} FXSA graphs for the time series.")\nif fxsa_graph_sequence:\n    print("Example FXSA graph data object:")\n    print(fxsa_graph_sequence[0])

## 6. FXSA GNN Model Implementation\n\nThis section defines the GNN for the FXSA task (`g_S`). As per the paper, it uses the same architecture as the FXRP GNN, but operates on the influence graph.

In [ ]:
# The FXSA_GNN has the same architecture as the FXRP_GNN.\nFXSA_GNN = FXRP_GNN \n\n# To instantiate it, we would need the feature dimensions from the FXSA graph sequence.\nif fxsa_graph_sequence:\n    fxsa_model = FXSA_GNN(num_node_features=fxsa_graph_sequence[0].num_node_features, \n                          num_edge_features=fxsa_graph_sequence[0].num_edge_features, \n                          hidden_dim=64, \n                          num_layers=3)\n    print("FXSA GNN Model Architecture:")\n    print(fxsa_model)

## 7. Constraint Handling Function (h_SO)\n\nThis section implements the post-processing function `h_SO`, which ensures that the output of the FXSA GNN satisfies the trading constraints.

In [ ]:
def post_process_trading_quantities(u_prime, P_hat):\n    """\n    This function takes the raw output of the FXSA GNN and applies the projection\n    and normalization to get the final trading quantities.\n    """\n    # Project onto the valid flow space\n    u = torch.matmul(P_hat, u_prime.unsqueeze(-1)).squeeze(-1)\n    \n    # Apply ReLU and normalize\n    u_pos = F.relu(u)\n    sum_u_pos = torch.sum(u_pos, dim=-1, keepdim=True)\n    w = u_pos / (sum_u_pos + 1e-8) # Add epsilon to avoid division by zero\n    \n    return w

## 8. Next Steps\n\nThis notebook has successfully fetched, preprocessed, and structured the data for both the FXRP and FXSA models. The GNN architectures and the constraint handling function are also implemented. The final steps are:\n\n1.  **Run FXRP Model:** Get predictions `X_hat` from the `FXRP_GNN`.\n2.  **Generate FXSA Graph:** Use the predictions to generate the FXSA graph sequence.\n3.  **Implement Training Loops:** Train both the `FXRP_GNN` and the `FXSA_GNN`.

## 9. Model Training and Execution Pipeline

### 9.1. Part 1: Train the FXRP Model

In [ ]:
import torch.optim as optim\nfrom torch.utils.data import random_split\n\n# First, regenerate the graph sequence with the correct targets\nall_currencies_for_fxrp = ['EUR', 'JPY', 'GBP', 'AUD', 'CAD', 'CHF', 'HKD', 'SGD', 'SEK']\ngraph_sequence_with_targets = generate_graph_sequence(df_full, all_currencies_for_fxrp, lookback_windows)\n\n# Split data into training and validation sets (e.g., 80/20 split)\ntrain_size = int(0.8 * len(graph_sequence_with_targets))\nval_size = len(graph_sequence_with_targets) - train_size\ntrain_dataset, val_dataset = random_split(graph_sequence_with_targets, [train_size, val_size])\n\nprint(f"Training set size: {len(train_dataset)}")\nprint(f"Validation set size: {len(val_dataset)}")\n\n# Instantiate the model\nfxrp_model = FXRP_GNN(num_node_features=train_dataset[0].num_node_features, \n                      num_edge_features=train_dataset[0].num_edge_features, \n                      hidden_dim=64, \n                      num_layers=3)\n\noptimizer = optim.Adam(fxrp_model.parameters(), lr=0.001)\nloss_fn = nn.MSELoss()\n\n# --- Training Loop ---\nnum_epochs = 10\nprint("\nStarting FXRP Model Training...")\nfor epoch in range(num_epochs):\n    fxrp_model.train()\n    total_loss = 0\n    for data in train_dataset:\n        optimizer.zero_grad()\n        # The model predicts the log return for each edge\n        pred_log_return = fxrp_model(data)\n        \n        # Calculate loss against the true log return\n        loss = loss_fn(pred_log_return, data.y)\n        \n        loss.backward()\n        optimizer.step()\n        total_loss += loss.item()\n    \n    avg_loss = total_loss / len(train_dataset)\n    print(f"Epoch {epoch+1}/{num_epochs}, Average Training Loss: {avg_loss:.8f}")\n\nprint("FXRP Model Training Finished.")

### 9.2. Part 2: Generate Predictions from Trained FXRP Model

In [ ]:
print("\nGenerating predictions for FXSA model...")\nfxrp_model.eval()\npredictions_list = []\n\nwith torch.no_grad():\n    for data in graph_sequence_with_targets:\n        # Get predicted log returns\n        pred_log_returns = fxrp_model(data).squeeze()\n        \n        # Get current day's FX rates from the full dataframe\n        current_rates = df_full.loc[data.timestamp]\n        \n        # Store predictions for this timestamp\n        prediction_row = {}\n        \n        # Apply the inverse transform h_PO: X_hat_t = X_{t-1} * exp(y_hat_t)\n        for i, ticker in enumerate(data.edge_tickers):\n            if ticker in current_rates:\n                predicted_rate = current_rates[ticker] * np.exp(pred_log_returns[i].item())\n                prediction_row[ticker] = predicted_rate\n            else:\n                # If we couldn't get a current rate, we can't predict\n                prediction_row[ticker] = 0\n\n        predictions_list.append(pd.Series(prediction_row, name=data.timestamp))\n\n# Create the predictions DataFrame\npredictions_df = pd.DataFrame(predictions_list)\npredictions_df = predictions_df.fillna(0) # Fill any NaNs that might have slipped through\n\nprint("Predictions DataFrame created for FXSA stage.")\nprint(predictions_df.head())

### 9.3. Part 3: Train the FXSA Model

In [ ]:
def fxsa_loss_fn(w_batch, ground_truth_batch, all_currencies, o='USD'):\n    \"""\n    Calculates the loss for the FXSA model based on Eq. 19 from the paper.\n    w_batch: A tensor of trading weights [batch_size, num_u_nodes].\n    ground_truth_batch: A list of pandas Series, each containing the true future FX rates for a timestep.\n    \"""\n    gains = []\n    \n    for i, w in enumerate(w_batch):\n        ground_truth = ground_truth_batch[i]\n        u_nodes = ground_truth['u_nodes']\n        H_ti = {c: 0.0 for c in all_currencies}\n\n        # Calculate realized holdings H_ti (Eq. 22)\n        for j, (c1, c2) in enumerate(u_nodes):\n            w_t_c1c2 = w[j].item()\n            if w_t_c1c2 > 0:\n                # Convert amount to trade from home currency 'o'\n                X_hat_oc1 = ground_truth['predictions'].get(f'{o}{c1}=X', 1.0) if c1 != o else 1.0\n                amount_in_c1 = X_hat_oc1 * w_t_c1c2\n                \n                # Get the *realized* exchange rate X_t_c2c1\n                realized_X_c2c1 = ground_truth['realized_rates'].get(f'{c2}{c1}=X', 0)\n                \n                # Inflow to c2\n                inflow_to_c2 = amount_in_c1 * realized_X_c2c1\n                H_ti[c2] += inflow_to_c2\n                # Outflow from c1\n                H_ti[c1] -= amount_in_c1\n\n        # Calculate total gain G_t (Eq. 21)\n        G_t = 0\n        for currency in all_currencies:\n             # Using simplified version from paper: assume interest factor is 1 for loss calculation\n            realized_X_co = ground_truth['future_rates'].get(f'{currency}{o}=X', 1.0) if currency != o else 1.0\n            G_t += H_ti[currency] * realized_X_co\n        \n        gains.append(G_t)\n\n    gains_tensor = torch.tensor(gains, dtype=torch.float)\n    \n    # Calculate mu_G and sigma_G^2\n    mu_G = torch.mean(gains_tensor)\n    sigma_G_sq = torch.var(gains_tensor, unbiased=True)\n    \n    # Avoid division by zero\n    if sigma_G_sq < 1e-8:\n        sigma_G_sq = 1e-8\n        \n    # Implement the conditional loss from Eq. 19\n    if mu_G > 0:\n        loss = -(mu_G**2 / sigma_G_sq)\n    else:\n        loss = -mu_G\n        \n    return loss

In [ ]:
# First, generate the FXSA graph sequence using the predictions from our trained FXRP model\nall_currencies_for_fxsa = ['USD', 'EUR', 'JPY', 'GBP', 'AUD', 'CAD', 'CHF', 'HKD', 'SGD', 'SEK']\nfxsa_graph_sequence = generate_fxsa_graph_sequence(predictions_df, all_currencies_for_fxsa, lookback_windows)\n\n# Split data\ntrain_size = int(0.8 * len(fxsa_graph_sequence))\nval_size = len(fxsa_graph_sequence) - train_size\nfxsa_train_dataset, fxsa_val_dataset = random_split(fxsa_graph_sequence, [train_size, val_size])\n\n# Instantiate the model\nif fxsa_train_dataset:\n    first_graph = fxsa_train_dataset[0]\n    fxsa_model = FXSA_GNN(num_node_features=first_graph.num_node_features, \n                          num_edge_features=first_graph.num_edge_features, \n                          hidden_dim=64, \n                          num_layers=3)\n\n    optimizer_fxsa = optim.Adam(fxsa_model.parameters(), lr=0.001)\n\n    # --- FXSA Training Loop ---\n    print("\nStarting FXSA Model Training...")\n    # This loop is more complex as the loss depends on a batch of results\n    batch_size = 32\n    for epoch in range(num_epochs):\n        fxsa_model.train()\n        total_loss = 0\n        \n        # Manual batching\n        for i in range(0, len(fxsa_train_dataset), batch_size):\n            batch = fxsa_train_dataset[i:i+batch_size]\n            if not batch: continue\n\n            optimizer_fxsa.zero_grad()\n            \n            u_prime_batch = []\n            P_hat_batch = []\n            ground_truth_batch = []\n\n            for data in batch:\n                u_prime = fxsa_model(data) # This is the raw output\n                u_prime_batch.append(u_prime.squeeze())\n                P_hat_batch.append(data.P_hat)\n                \n                # Prepare ground truth data needed for the loss function\n                timestamp = data.timestamp\n                next_timestamp = df_full.index[df_full.index.get_loc(timestamp) + 1]\n                \n                ground_truth_batch.append({\n                    'predictions': predictions_df.loc[timestamp], # X_hat for calculating trade amount\n                    'realized_rates': df_full.loc[timestamp],   # X_t for H_ti\n                    'future_rates': df_full.loc[next_timestamp], # X_{t+1} for G_t\n                    'u_nodes': data.u_nodes\n                })\n\n            # Process the batch\n            w_batch = []\n            for u_prime, P_hat in zip(u_prime_batch, P_hat_batch):\n                w = post_process_trading_quantities(u_prime, P_hat)\n                w_batch.append(w)\n            \n            w_batch_tensor = torch.stack(w_batch)\n\n            loss = fxsa_loss_fn(w_batch_tensor, ground_truth_batch, all_currencies_for_fxsa)\n            \n            loss.backward()\n            optimizer_fxsa.step()\n            total_loss += loss.item()\n\n        avg_loss = total_loss / (len(fxsa_train_dataset) / batch_size)\n        print(f"Epoch {epoch+1}/{num_epochs}, Average Training Loss: {avg_loss:.8f}")\n\n    print("FXSA Model Training Finished.")\nelse:\n    print("FXSA dataset is empty. Skipping training.")

## 10. Model Evaluation

### 10.1. Evaluating the FXRP (Prediction) Model

In [ ]:
# --- Calculate FXRP Model MSE on the validation set ---\nfxrp_model.eval()\ntotal_val_loss = 0\n\nwith torch.no_grad():\n    for data in val_dataset: # Using the validation set from the FXRP training split\n        pred_log_return = fxrp_model(data)\n        loss = loss_fn(pred_log_return, data.y)\n        total_val_loss += loss.item()\n\navg_val_mse = total_val_loss / len(val_dataset)\nprint(f"FXRP Model - Mean Squared Error (MSE) on Validation Set: {avg_val_mse:.8f}")

### 10.2. Backtesting and Evaluating the FXSA (Arbitrage) Model

In [ ]:
# --- Backtest the FXSA Strategy on the validation set ---\nfxsa_model.eval()\ndaily_pnl = []\n\nwith torch.no_grad():\n    for data in fxsa_val_dataset:\n        # 1. Get the raw model output (u_prime)\n        u_prime = fxsa_model(data).squeeze()\n        \n        # 2. Get the final trading weights (w) using the post-processing function\n        w = post_process_trading_quantities(u_prime, data.P_hat)\n        \n        # 3. Calculate the realized P&L for this day\n        # We use the same logic as the loss function's gain calculation (G_t)\n        timestamp = data.timestamp\n        next_timestamp = df_full.index[df_full.index.get_loc(timestamp) + 1]\n        \n        # Get the ground truth data for this day\n        ground_truth = {\n            'predictions': predictions_df.loc[timestamp],\n            'realized_rates': df_full.loc[timestamp],\n            'future_rates': df_full.loc[next_timestamp],\n            'u_nodes': data.u_nodes\n        }\n\n        # This inner logic is taken directly from the fxsa_loss_fn\n        H_ti = {c: 0.0 for c in all_currencies_for_fxsa}\n        for j, (c1, c2) in enumerate(ground_truth['u_nodes']):\n            w_t_c1c2 = w[j].item()\n            if w_t_c1c2 > 0:\n                o = 'USD'\n                X_hat_oc1 = ground_truth['predictions'].get(f'{o}{c1}=X', 1.0) if c1 != o else 1.0\n                amount_in_c1 = X_hat_oc1 * w_t_c1c2\n                realized_X_c2c1 = ground_truth['realized_rates'].get(f'{c2}{c1}=X', 0)\n                inflow_to_c2 = amount_in_c1 * realized_X_c2c1\n                H_ti[c2] += inflow_to_c2\n                H_ti[c1] -= amount_in_c1\n\n        G_t = 0\n        for currency in all_currencies_for_fxsa:\n            o = 'USD'\n            realized_X_co = ground_truth['future_rates'].get(f'{currency}{o}=X', 1.0) if currency != o else 1.0\n            G_t += H_ti[currency] * realized_X_co\n        \n        daily_pnl.append(G_t)\n\npnl_series = pd.Series(daily_pnl)\n\n# --- Calculate Financial Metrics ---\nif not pnl_series.empty:\n    # Assuming 252 trading days in a year\n    annualization_factor = np.sqrt(252)\n\n    # Information Ratio (similar to Sharpe Ratio, assuming zero benchmark return)\n    mean_daily_return = pnl_series.mean()\n    std_daily_return = pnl_series.std()\n    information_ratio = (mean_daily_return / std_daily_return) * annualization_factor\n\n    # Annual Volatility\n    annual_volatility = std_daily_return * annualization_factor\n\n    # Maximum Drawdown (MDD)\n    cumulative_returns = (1 + pnl_series).cumprod()\n    peak = cumulative_returns.expanding(min_periods=1).max()\n    drawdown = (cumulative_returns - peak) / peak\n    max_drawdown = drawdown.min()\n\n    print("\n--- FXSA Model Backtest Results ---")\n    print(f"Information Ratio (Annualized): {information_ratio:.4f}")\n    print(f"Annual Volatility: {annual_volatility:.4%}")\n    print(f"Maximum Drawdown: {max_drawdown:.4%}")\nelse:\n    print("Could not generate P&L series for backtesting.")